# Import libraries

* Import the necessary libraries including Pandas, Plotly, and others.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

plt.rcParams['figure.figsize'] = (12,6)
plt.style.use('fivethirtyeight')

import warnings
warnings.filterwarnings("ignore")

from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.model_selection import GridSearchCV

from sklearn.pipeline import make_pipeline

# Load the Dataset

* Load the dataset into a Pandas DataFrame.


In [2]:
df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')

In [3]:
df.head(5).style.set_properties(**{'background-color':'lightblue','color':'black','border-color':'#8b8c8c'})

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.000000,False,0.000000,0.000000,0.000000,0.000000,0.000000,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.000000,False,109.000000,9.000000,25.000000,549.000000,44.000000,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.000000,True,43.000000,3576.000000,0.000000,6715.000000,49.000000,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.000000,False,0.000000,1283.000000,371.000000,3329.000000,193.000000,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.000000,False,303.000000,70.000000,151.000000,565.000000,2.000000,Willy Santantines,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


### Check for the presence of missing values in the dataset and handle them appropriately.



In [5]:
# Check for the presence of missing values
df.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
df['HomePlanet'].value_counts()

Earth     4602
Europa    2131
Mars      1759
Name: HomePlanet, dtype: int64

In [7]:
df['Cabin'] = df['Cabin'].fillna('0')

In [8]:
df['Cabin'] = df['Cabin'].str.split('/')
df['Cabin']

0          [B, 0, P]
1          [F, 0, S]
2          [A, 0, S]
3          [A, 0, S]
4          [F, 1, S]
            ...     
8688      [A, 98, P]
8689    [G, 1499, S]
8690    [G, 1500, S]
8691     [E, 608, S]
8692     [E, 608, S]
Name: Cabin, Length: 8693, dtype: object

In [9]:
df['Cabin'].isnull().sum()

0

In [10]:
def get_cabin(x):
  print(x)
  print(type(x))
  print(type(x) == np.float32)
  print('nan', x == np.nan)
  if type(x) == 'float':
    return np.nan
  elif len(x) > 1:
    return x[0]
  else:
    return x[0]

In [11]:
df['Cabin_zone'] = df['Cabin'].apply(get_cabin)

['B', '0', 'P']
<class 'list'>
False
nan False
['F', '0', 'S']
<class 'list'>
False
nan False
['A', '0', 'S']
<class 'list'>
False
nan False
['A', '0', 'S']
<class 'list'>
False
nan False
['F', '1', 'S']
<class 'list'>
False
nan False
['F', '0', 'P']
<class 'list'>
False
nan False
['F', '2', 'S']
<class 'list'>
False
nan False
['G', '0', 'S']
<class 'list'>
False
nan False
['F', '3', 'S']
<class 'list'>
False
nan False
['B', '1', 'P']
<class 'list'>
False
nan False
['B', '1', 'P']
<class 'list'>
False
nan False
['B', '1', 'P']
<class 'list'>
False
nan False
['F', '1', 'P']
<class 'list'>
False
nan False
['G', '1', 'S']
<class 'list'>
False
nan False
['F', '2', 'P']
<class 'list'>
False
nan False
['0']
<class 'list'>
False
nan False
['F', '3', 'P']
<class 'list'>
False
nan False
['F', '4', 'P']
<class 'list'>
False
nan False
['F', '5', 'P']
<class 'list'>
False
nan False
['G', '0', 'P']
<class 'list'>
False
nan False
['F', '6', 'P']
<class 'list'>
False
nan False
['E', '0', 'S']
<class 

In [12]:
import plotly.express as px
import plotly.graph_objects as go

# define function for random color palette
def r_color(num=2, seed=None):
    pal = px.colors.qualitative.Pastel
    if seed==None:
        seed = np.random.randint(0,420,size=1)
    np.random.seed(seed)
    print(seed)
    return [pal[i] for i in np.random.randint(0,5,size=num)]

# define function for count plot
def count_plot(column, seed=None, num=3, transported=False):
    # create figure
    fig = go.Figure()
    # add trace for count plot
    fig.add_trace(px.histogram(df, x=column, color='Transported', color_discrete_sequence=r_color(num, seed=seed)).data[0])
    # set layout
    fig.update_layout(
        title=f'Count of {column}',
        xaxis_title='Count',
        yaxis_title=column,
        font=dict(
            size=18
        )
    )
    # show figure
    fig.show()
    # print value counts
    print('value_counts\n',df[column].value_counts())
    print('')
    print('')
    if transported:
        # print transported grouped by
        print('transported grouped by \n', df.groupby(column)['Transported'].value_counts())

# fill missing values in Cabin column with '0'
df['Cabin'] = df['Cabin'].fillna('0')
# split Cabin column values by '/'
df['Cabin'] = df['Cabin'].str.split('/')
# apply function to get the first value from the list or NaN if the value is a float
df['Cabin'] = df['Cabin'].apply(lambda x: x[0] if type(x) == list and len(x) > 0 else np.nan if type(x) == float else x)

# call count_plot function for each column
for col in ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
    count_plot(col)

[131]


value_counts
 Earth     4602
Europa    2131
Mars      1759
Name: HomePlanet, dtype: int64


[409]


value_counts
 False    5439
True     3037
Name: CryoSleep, dtype: int64


[163]


value_counts
 Series([], Name: Cabin, dtype: int64)


[307]


value_counts
 TRAPPIST-1e      5915
55 Cancri e      1800
PSO J318.5-22     796
Name: Destination, dtype: int64


[317]


...

value_counts
 0.0       5577
1.0        117
2.0         79
3.0         61
4.0         47
          ... 
1612.0       1
2598.0       1
632.0        1
378.0        1
745.0        1
Name: RoomService, Length: 1273, dtype: int64


[242]


value_counts
 0.0       5456
1.0        116
2.0         75
3.0         53
4.0         53
          ... 
3846.0       1
5193.0       1
312.0        1
827.0        1
4688.0       1
Name: FoodCourt, Length: 1507, dtype: int64


[36]


value_counts
 0.0       5587
1.0        153
2.0         80
3.0         59
4.0         45
          ... 
3627.0       1
2074.0       1
871.0        1
742.0        1
1872.0       1
Name: ShoppingMall, Length: 1115, dtype: int64


[250]


value_counts
 0.0       5324
1.0        146
2.0        105
5.0         53
3.0         53
          ... 
273.0        1
2581.0       1
2948.0       1
3778.0       1
1643.0       1
Name: Spa, Length: 1327, dtype: int64


[58]


value_counts
 0.0       5495
1.0        139
2.0         70
3.0         56
5.0         51
          ... 
408.0        1
876.0        1
2891.0       1
2102.0       1
3235.0       1
Name: VRDeck, Length: 1306, dtype: int64




In [13]:
fig = px.histogram(df, x="HomePlanet", color="Transported", barmode="stack")
fig.show()

### Check the distribution of the target variable "Transported" using a count plot created with Plotly.

In [14]:
# Create a count plot to show the distribution of the target variable
fig = px.histogram(df, x='Transported', color='Transported', barmode='group')
fig.show()

In [15]:
# define function for random color palette
def r_color(num=2, seed=None):
    pal = px.colors.qualitative.Pastel
    if seed==None:
        seed = np.random.randint(0,420,size=1)
    np.random.seed(seed)
    print(seed)
    return [pal[i] for i in np.random.randint(0,5,size=num)]

# define function for bivariate plot
def bivariate_plot(column):
    df_tmp1 = pd.DataFrame(df[column].value_counts() / sum(df[column].value_counts()))
    df_tmp2 = pd.DataFrame(df[df['Transported'] == 1 ][column].value_counts() / sum(df[df['Transported'] == 1 ][column].value_counts()))
    df_tmp3 = df_tmp2 - df_tmp1
    df_final = pd.concat([df_tmp1,df_tmp2,df_tmp3], axis=1)
    df_final.columns = [f'% {column} Start', f'% {column} Transp\'d', 'Difference']
    fig = px.histogram(df, x=column, color='Transported', barmode='group', color_discrete_sequence=r_color(2))
    fig.update_layout(
        title=f'{column} vs Transported',
        xaxis_title=column,
        yaxis_title='Transported',
        font=dict(
            size=18
        )
    )
    fig.show()
    fig = px.histogram(df_final, x='Difference', color_discrete_sequence=r_color(2))
    fig.update_layout(
        title=f'{column} Difference in Proportion',
        xaxis_title='Difference in Proportion',
        yaxis_title='Transported',
        font=dict(
            size=18
        )
    )
    fig.show()

# call bivariate_plot function for each column
for col in ['HomePlanet', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
    bivariate_plot(col)


[398]


[29]


[34]


[361]


[248]


...

[306]


[281]


[143]


[33]


[66]


In [16]:
# Create a scatter plot to show the relationship between age and spa billing amount
fig = px.scatter(df, x='Age', y='Spa', color='Transported')
fig.show()

### Create a scatter plot using Plotly to show the relationship between the age of passengers and the amount they billed at the spa on the Spaceship Titanic.

In [17]:
transported = df[df['Transported'] == True]
not_transported = df[df['Transported'] == False]

fig = px.histogram(df, x="Age", color="Transported", nbins=20,
                   labels={'Age':'Passenger Age', 'count': 'Number of Passengers'},
                   title="Distribution of Passenger Ages by Transported Status")
fig.show()

### Create a heatmap using Plotly to show the correlation between the different amenities on the Spaceship Titanic.

Add color to the heatmap to show the strength of the correlation between the amenities.



In [18]:
import plotly.figure_factory as ff

corr = df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].corr()
corr_matrix = np.array(corr)

fig = ff.create_annotated_heatmap(z=corr_matrix, x=list(corr.columns), y=list(corr.columns),
                                  annotation_text=corr_matrix.round(2),
                                  colorscale='Viridis', showscale=True,
                                  xgap=1, ygap=1)
fig.update_layout(title='Correlation Between Different Amenities on Spaceship Titanic')
fig.show()

### Create a pair plot using Plotly to show the pairwise relationships between all the numeric variables in the dataset.

Add color to the pair plot to distinguish between passengers who were transported to another dimension and those who were not.

In [19]:
fig = px.scatter_matrix(df, dimensions=['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'],
                        color='Transported', title="Pairwise Relationships Between Numeric Variables")
fig.show()


Add color to the box plot to distinguish between the two groups.

In [20]:
df["BilledAmount"] = df["VIP"]*100 + df["RoomService"]*10 + df["FoodCourt"]*5 + df["ShoppingMall"]*20 + df["Spa"]*50 + df["VRDeck"]*30


In [21]:
fig = px.box(df, x="Destination", y="BilledAmount", color="Transported")
fig.update_layout(title="Distribution of Billed Amounts for Amenities by Transported Status")
fig.show()

### Create a histogram using Plotly to show the distribution of passenger ages for those who were transported to another dimension and those who were not.

In [22]:
fig = px.histogram(df, x="Age", color="Transported")
fig.update_layout(title="Distribution of Passenger Ages by Transported Status")
fig.show()


# Preprocessing:

Before we start with the modeling, we need to preprocess the data. We will first check for null values in the dataset and then perform feature scaling.

In [23]:
# Check for null values
print(df.isnull().sum())

PassengerId        0
HomePlanet       201
CryoSleep        217
Cabin           8693
Destination      182
Age              179
VIP              203
RoomService      181
FoodCourt        183
ShoppingMall     208
Spa              183
VRDeck           188
Name             200
Transported        0
Cabin_zone         0
BilledAmount    1096
dtype: int64


In [24]:
df[(df['HomePlanet'].isnull()) & (df['CryoSleep'].isnull())]

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Cabin_zone,BilledAmount
3622,3896_01,NaN,NaN,NaN,55 Cancri e,18.0,False,0.0,4387.0,0.0,2241.0,0.0,Ainoxa Preeldy,True,C,133985.0
7218,7711_01,NaN,NaN,NaN,TRAPPIST-1e,24.0,False,0.0,82.0,0.0,1624.0,77.0,Jihoton Muspereed,False,D,83920.0


In [25]:
df[(df['HomePlanet'].isnull()) & (df['CryoSleep'].isnull())].index

Int64Index([3622, 7218], dtype='int64')

In [26]:
df[(df['HomePlanet'].isnull()) & (df['CryoSleep'].isnull()) & (df['Cabin'].isnull())]

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Cabin_zone,BilledAmount
3622,3896_01,NaN,NaN,NaN,55 Cancri e,18.0,False,0.0,4387.0,0.0,2241.0,0.0,Ainoxa Preeldy,True,C,133985.0
7218,7711_01,NaN,NaN,NaN,TRAPPIST-1e,24.0,False,0.0,82.0,0.0,1624.0,77.0,Jihoton Muspereed,False,D,83920.0


In [27]:
inds_drop = []
for col1 in df.columns:
  for col2 in df.columns:
    if col1 != col2:
      for col3 in df.columns:
        if col2 != col3:
          inds = df[(df[col1].isnull()) & (df[col2].isnull()) & (df[col3].isnull())].index
          print(col1,col2,col3,inds)
          for i in inds:
            inds_drop.append(i)
len(inds_drop)

PassengerId HomePlanet PassengerId Int64Index([], dtype='int64')
PassengerId HomePlanet CryoSleep Int64Index([], dtype='int64')
PassengerId HomePlanet Cabin Int64Index([], dtype='int64')
PassengerId HomePlanet Destination Int64Index([], dtype='int64')
PassengerId HomePlanet Age Int64Index([], dtype='int64')
PassengerId HomePlanet VIP Int64Index([], dtype='int64')
PassengerId HomePlanet RoomService Int64Index([], dtype='int64')
PassengerId HomePlanet FoodCourt Int64Index([], dtype='int64')
PassengerId HomePlanet ShoppingMall Int64Index([], dtype='int64')
PassengerId HomePlanet Spa Int64Index([], dtype='int64')
PassengerId HomePlanet VRDeck Int64Index([], dtype='int64')
PassengerId HomePlanet Name Int64Index([], dtype='int64')
PassengerId HomePlanet Transported Int64Index([], dtype='int64')
PassengerId HomePlanet Cabin_zone Int64Index([], dtype='int64')
PassengerId HomePlanet BilledAmount Int64Index([], dtype='int64')
PassengerId CryoSleep PassengerId Int64Index([], dtype='int64')
Passen

19314

In [28]:
print(df.shape)
df = df.drop(inds_drop)
df.shape

(8693, 16)


(6764, 16)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6764 entries, 0 to 8692
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   6764 non-null   object 
 1   HomePlanet    6764 non-null   object 
 2   CryoSleep     6764 non-null   object 
 3   Cabin         0 non-null      float64
 4   Destination   6764 non-null   object 
 5   Age           6764 non-null   float64
 6   VIP           6764 non-null   object 
 7   RoomService   6764 non-null   float64
 8   FoodCourt     6764 non-null   float64
 9   ShoppingMall  6764 non-null   float64
 10  Spa           6764 non-null   float64
 11  VRDeck        6764 non-null   float64
 12  Name          6764 non-null   object 
 13  Transported   6764 non-null   bool   
 14  Cabin_zone    6764 non-null   object 
 15  BilledAmount  6764 non-null   object 
dtypes: bool(1), float64(7), object(8)
memory usage: 852.1+ KB


In [30]:
df['Cabin'].value_counts().index

Float64Index([], dtype='float64')

In [31]:
for feat in df.columns[:-2]: #not cabin
  if feat not in ['Cabin', 'Cabin_zone', 'Cabin_code']:
    if df[feat].dtype == 'float':
      df[feat] = df[feat].fillna(df[feat].median())
    elif df[feat].dtype == 'object':
      
      df[feat] = df[feat].fillna(df[feat].value_counts().index[0])

In [32]:
df.isnull().sum()

PassengerId        0
HomePlanet         0
CryoSleep          0
Cabin           6764
Destination        0
Age                0
VIP                0
RoomService        0
FoodCourt          0
ShoppingMall       0
Spa                0
VRDeck             0
Name               0
Transported        0
Cabin_zone         0
BilledAmount       0
dtype: int64

# Treat Outliers:

We can check for outliers using a boxplot. If we find any outliers, we can remove them using the IQR method.



In [39]:
import plotly.express as px
import plotly.graph_objs as go

def plot(column, bins=30, hue=None, num=3):
    # Plot histogram
    fig_hist = px.histogram(df, x=column, color=hue, nbins=bins)
    fig_hist.update_layout(title={
        'text': f"{column} Histogram",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': 'black', 'family':'Arial'}
    })
    fig_hist.show()

    # Identify outliers
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    lower_bound = q1 - 1.5 * iqr

    # Plot box plot with outliers
    fig_box = go.Figure()
    fig_box.add_trace(go.Box(y=df[column], name=column, boxpoints='outliers'))
    fig_box.update_layout(title={
        'text': f"{column} Box Plot",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 22, 'color': 'black', 'family':'Arial'}
    })
    fig_box.add_shape(
        type='line', x0=upper_bound, y0=0, x1=upper_bound, y1=1,
        line=dict(color='red', width=2, dash='dash')
    )
    fig_box.add_shape(
        type='line', x0=lower_bound, y0=0, x1=lower_bound, y1=1,
        line=dict(color='red', width=2, dash='dash')
    )
    fig_box.show()


# Dimension Reduction using PCA

In [41]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder


# create sample dataframe
df = pd.DataFrame({
    'Gender': ['Male', 'Female', 'Male', 'Female'],
    'Age': [25, 30, 35, 40],
    'RoomService': [8, 9, 2, 5],
    'FoodCourt': [5, 6, 8, 9],
    'ShoppingMall': [7, 8, 9, 6],
    'Spa': [6, 5, 9, 8],
    'VRDeck': [4, 6, 8, 7],
    'Transported': ['True', 'False', 'False', 'True']
})

# perform one-hot encoding on categorical columns
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df_encoded = pd.get_dummies(df, columns=['Transported'])

# perform one-hot encoding on 'Transported'
df_encoded[['Transported_False', 'Transported_True']] = pd.get_dummies(df['Transported'])

# create a new DataFrame with the actual `Transported` values
df_transported = df_encoded.drop(['Transported_False', 'Transported_True'], axis=1)
df_transported['Transported'] = df['Transported'].replace({'False': 0, 'True': 1})

# show encoded dataframe
df_encoded.head()


,Gender,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported_False,Transported_True
0,1,25,8,5,7,6,4,0,1
1,0,30,9,6,8,5,6,1,0
2,1,35,2,8,9,9,8,1,0
3,0,40,5,9,6,8,7,0,1


In [42]:
def remove_outliers(df, columns=None):
    """
    Removes outliers from a Pandas DataFrame.
    :param df: the DataFrame to remove outliers from
    :param columns: the list of columns to check for outliers (default is all columns)
    :return: the DataFrame with outliers removed
    """
    if columns is None:
        columns = df.columns
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        df = df[(df[col] >= q1 - 1.5*iqr) & (df[col] <= q3 + 1.5*iqr)]
    return df


In [43]:
# remove outliers from the dataframe
df = remove_outliers(df_encoded)

# perform PCA on the cleaned dataframe
pca = PCA(n_components=2)
principal_components = pca.fit_transform(df.drop(['Transported_False', 'Transported_True'], axis=1))


In [44]:
# create the original DataFrame
data = {
    'Gender': [1, 0, 1, 0],
    'Age': [25, 30, 35, 40],
    'RoomService': [8, 9, 2, 5],
    'FoodCourt': [5, 6, 8, 9],
    'ShoppingMall': [7, 8, 9, 6],
    'Spa': [6, 5, 9, 8],
    'VRDeck': [4, 6, 8, 7],
    'Transported_False': [0, 1, 1, 0],
    'Transported_True': [1, 0, 0, 1]
}

df = pd.DataFrame(data)

# Rename columns
df = df.rename(columns={'Transported_False': 'Transported_0', 'Transported_True': 'Transported_1'})

# Create a new column with the actual `Transported` values
df['Transported'] = df['Transported_1']

# Drop unnecessary columns
df = df.drop(['Transported_0', 'Transported_1'], axis=1)

# Show DataFrame
df.head()


,Gender,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,1,25,8,5,7,6,4,1
1,0,30,9,6,8,5,6,0
2,1,35,2,8,9,9,8,0
3,0,40,5,9,6,8,7,1


In [45]:
df_pca = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
df_pca['Transported'] = df['Transported']

fig = px.scatter(df_pca, x='PC1', y='PC2', color='Transported')
fig.show()